# Simple RNN · Official R3-3 Model (Sequence-Averaged Simple RNN)

This notebook defines, inspects, and documents the **official selected Simple RNN architecture (Candidate R3-3)** for hotel review sentiment classification.

- **Architecture**: `Embedding(5000, 64) → SimpleRNN(64, return_sequences=True, dropout=0.5) → GlobalAveragePooling1D() → Dense(32, ReLU) → Dropout(0.3) → Dense(1, Sigmoid)`
- **Parameters**: 330,369 trainable parameters (identical footprint to the initial un-gated baseline)
- **Selection Basis**: Validated in Round 3 hyperparameter experiments (`RNN_Hyperparameter_Experiments.ipynb`) strictly on training + validation splits (Val Accuracy: **0.9438**, Val Loss: **0.137641**, Val F1: **0.943396**)
- **Final Test Performance**: **95.00% Accuracy**, **0.979531 ROC-AUC**, **0.9500 F1** on the untouched 160-review test set
- **Checkpoint**: Loads the verified official weights from `simple_rnn/results/simple_rnn_model.keras` without retraining.

## 1. Setup & Project Environment

Each notebook in this project is self-contained. The cell below configures paths to the project root, dataset, and results folder, importing standard numerical and deep learning dependencies.

In [ ]:
# ---- Imports, project folders and dataset location ----------------------------------------------
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

layers = keras.layers
IN_COLAB = "google.colab" in sys.modules

# RNN_ROOT is the folder that holds the numbered step folders (01_Setup, 02_Data_Audit, ...).
here = pathlib.Path.cwd().resolve()
RNN_ROOT = next((p for p in [here, *here.parents] if (p / "01_Setup").is_dir() or (p / "simple_rnn").is_dir()), here)
if (RNN_ROOT / "simple_rnn").is_dir():
    RNN_ROOT = RNN_ROOT / "simple_rnn"
RESULTS = RNN_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# The dataset is the primary input file.
DATA_NAME = "deceptive-opinion.csv"
search_in = [RNN_ROOT, RNN_ROOT / "dataset", RNN_ROOT.parent / "dataset", here, here / "dataset",
             here.parent / "dataset", pathlib.Path("/content")]
DATA_PATH = next((d / DATA_NAME for d in search_in if (d / DATA_NAME).exists()), None)

if DATA_PATH is None and IN_COLAB:
    from google.colab import files
    print(f"Please choose {DATA_NAME} from your computer...")
    uploaded = files.upload()
    DATA_PATH = here / next(iter(uploaded))

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("Results folder:", RESULTS)
print("Dataset:", DATA_PATH if DATA_PATH else "Not found (only needed if reprocessing raw data)")

In [ ]:
# ---- Group settings: identical for Simple RNN, LSTM, GRU and 1D CNN ------------------------------
SEED = 42

# Labels: polarity column is the binary target (negative: 0, positive: 1)
LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

# Text -> integer sequence tokenisation
VOCAB_SIZE = 5000          # total ids, including <PAD>=0 and <OOV>=1
MAX_LEN = 300              # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre"            # zeros in front, so recurrent layer ends on real words
TRUNCATING = "post"        # long reviews keep their first MAX_LEN tokens

# Training and evaluation hyperparameters
EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5               # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5            # probability cut-off used for every model's test metrics

# Fixed 80/10/10 split (stratified on label, seed 42, made after removing duplicates).
VAL_IDS = {
    3, 4, 6, 7, 14, 24, 25, 27, 33, 38, 46, 50, 60, 66, 72, 74, 77, 85, 89, 92, 106, 110, 112, 136,
    144, 172, 211, 215, 224, 230, 238, 242, 248, 252, 288, 302, 311, 319, 336, 345, 368, 376, 385,
    393, 402, 404, 434, 440, 442, 465, 473, 510, 512, 515, 517, 524, 533, 536, 542, 549, 553, 558,
    579, 581, 594, 603, 624, 630, 638, 644, 645, 661, 663, 665, 699, 713, 729, 754, 760, 788, 801,
    813, 820, 821, 827, 832, 840, 864, 873, 893, 897, 900, 904, 907, 908, 923, 930, 936, 962, 963,
    989, 991, 1002, 1008, 1017, 1043, 1045, 1054, 1073, 1098, 1112, 1119, 1134, 1143, 1170, 1189,
    1191, 1201, 1239, 1258, 1262, 1263, 1264, 1270, 1280, 1295, 1308, 1309, 1314, 1328, 1344, 1350,
    1359, 1360, 1364, 1367, 1368, 1369, 1370, 1377, 1378, 1381, 1418, 1423, 1450, 1467, 1479, 1480,
    1490, 1505, 1506, 1522, 1547, 1551, 1552, 1573, 1578, 1588, 1593, 1594
}
TEST_IDS = {
    0, 2, 5, 9, 39, 45, 47, 55, 108, 117, 120, 125, 138, 139, 145, 171, 191, 193, 213, 222, 227,
    228, 232, 256, 258, 260, 272, 273, 276, 281, 299, 314, 321, 325, 349, 362, 364, 391, 409, 424,
    436, 445, 467, 469, 484, 502, 503, 505, 507, 523, 525, 531, 545, 547, 572, 573, 605, 606, 612,
    622, 623, 633, 669, 671, 681, 687, 691, 698, 718, 725, 731, 741, 752, 753, 765, 770, 771, 779,
    784, 785, 804, 814, 834, 835, 847, 852, 859, 866, 882, 887, 889, 901, 932, 940, 958, 961, 968,
    972, 993, 1003, 1007, 1019, 1020, 1033, 1044, 1046, 1055, 1072, 1076, 1079, 1091, 1117, 1141,
    1147, 1149, 1151, 1176, 1182, 1183, 1193, 1195, 1196, 1205, 1210, 1217, 1222, 1231, 1259, 1265,
    1275, 1278, 1279, 1288, 1293, 1296, 1302, 1306, 1312, 1324, 1357, 1365, 1366, 1396, 1404, 1416,
    1446, 1451, 1465, 1478, 1485, 1487, 1492, 1503, 1504, 1516, 1524, 1540, 1562, 1582, 1597
}
keras.utils.set_random_seed(SEED)
print("Locked split defined: 1,276 Train | 160 Val | 160 Test (Seed 42)")

## 2. The Official Simple RNN Architecture (R3-3: Sequence-Averaged)

### Theoretical Motivation & Architectural Design
The classic Elman Simple RNN recurrence relation is:
$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

Unlike LSTM or GRU, the Simple RNN lacks internal gating mechanisms (input, forget, or update gates) and has no additive cell state. Consequently:
1. **Vanishing Gradient & Recency Bias**: Across sequences of up to 300 tokens, backpropagating errors through repeated $W_{hh}$ multiplications causes exponential gradient decay. When only the final hidden state $h_T$ is fed to the classifier, the network suffers severe *recency bias*—disproportionately weighing the final 15–20 words and forgetting critical sentiments expressed earlier.
2. **Information Bottleneck**: Compressing up to 300 words into a single 64-dimensional vector $h_T$ creates an acute representation bottleneck.

### The R3-3 Solution: Sequence Averaging
In the selected **R3-3 architecture**, `SimpleRNN(64, return_sequences=True, dropout=0.5)` emits hidden state vectors for every token timestep:
$$(h_1, h_2, \dots, h_T) \in \mathbb{R}^{300 \times 64}$$

A non-parametric `GlobalAveragePooling1D()` layer then computes the arithmetic mean across the sequence dimension:
$$\bar{h} = \frac{1}{T} \sum_{t=1}^{T} h_t$$

This sequence-averaged vector $\bar{h}$ grants the downstream dense classifier direct, equal visibility into sentiment representations from all parts of the review. It neutralizes the single-state recency bottleneck **without introducing any gates or extra parameters**.

```
Input (300 token ids)
  → Embedding(5000, 64)                                320,000 params (5000 * 64)
  → SimpleRNN(64, return_sequences=True, dropout=0.5)    8,256 params (W_xh: 64*64 + W_hh: 64*64 + b: 64)
  → GlobalAveragePooling1D()                                 0 params (arithmetic mean over 300 timesteps)
  → Dense(32, ReLU)                                      2,080 params (64 * 32 + 32)
  → Dropout(0.3)                                             0 params (dense regularizer)
  → Dense(1, Sigmoid)                                       33 params (32 * 1 + 1)
Total Trainable Parameters: 330,369
```

| Layer | Output Shape | Param # | Configuration & Role |
|---|---|---|---|
| **Input** | `(None, 300)` | 0 | Sequence of 300 pre-padded integer token IDs. |
| **Embedding(5000, 64)** | `(None, 300, 64)` | 320,000 | Maps discrete token IDs to dense 64-d semantic vectors (96.86% of total params). |
| **SimpleRNN(64, return_sequences=True, dropout=0.5)** | `(None, 300, 64)` | 8,256 | Un-gated Elman recurrence with 0.5 input dropout; emits all hidden states $(h_1, \dots, h_{300})$. |
| **GlobalAveragePooling1D** | `(None, 64)` | 0 | Non-parametric temporal pooling across timesteps; eliminates single-state recency bias. |
| **Dense(32, ReLU)** | `(None, 32)` | 2,080 | Intermediate non-linear feature projection. |
| **Dropout(0.3)** | `(None, 32)` | 0 | Prevents dense co-adaptation on the small training set. |
| **Dense(1, Sigmoid)** | `(None, 1)` | 33 | Logistic output predicting $P(\text{positive}) \in [0, 1]$. |

In [ ]:
def build_simple_rnn():
    """Builds the official R3-3 Sequence-Averaged Simple RNN model."""
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, EMBED_DIM),
        layers.SimpleRNN(64, return_sequences=True, dropout=0.5),
        layers.GlobalAveragePooling1D(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ], name="simple_rnn_sentiment_r3_3")

model = build_simple_rnn()
model.summary()

In [ ]:
def n_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

breakdown = {layer.name: n_params(layer.trainable_weights) for layer in model.layers if layer.trainable_weights}
print("Trainable parameters per layer:")
for name, count in breakdown.items():
    print(f"  {name:25s}: {count:,} ({100 * count / model.count_params():.2f}%)")

print("-" * 50)
print(f"Total Trainable Parameters: {model.count_params():,}")
print(f"Recurrent Parameters (W_xh + W_hh + b): {breakdown['simple_rnn']:,} (64*64 + 64*64 + 64)")
print(f"Embedding Share: {100 * breakdown['embedding'] / model.count_params():.2f}%")

## 3. Official Trained Checkpoint Verification

Per the project protocol:
- We do **not** retrain or regenerate weights in this notebook.
- We load the official, verified checkpoint saved at `results/simple_rnn_model.keras` (which is identical to `results/r3_3_selected_model.keras`).

In [ ]:
model_path = RESULTS / "simple_rnn_model.keras"
if model_path.exists():
    official_model = keras.models.load_model(model_path)
    print(f"Successfully loaded official checkpoint: {model_path.name}")
    print(f"Trainable parameters: {official_model.count_params():,}")
    print("Checkpoint architecture summary:")
    official_model.summary()
else:
    print(f"Notice: {model_path} not found. Using initialized model definition.")

## 4. Controlled Tuning & Selection Process (Rounds 1–3)

The selection of R3-3 was conducted through a disciplined 3-stage empirical experimentation process documented in `RNN_Hyperparameter_Experiments.ipynb`. **Crucially, all model comparisons and decisions were made strictly on the 1,276 training and 160 validation reviews with zero test set leakage.**

### A. The Initial Baseline Model
- **Architecture**: `Embedding(5000, 64) → SimpleRNN(64, final step) → Dropout(0.5) → Dense(32) → Dropout(0.3) → Dense(1)`
- **Parameters**: 330,369
- **Validation Metrics**: Accuracy = 0.9250 (148/160), Loss = 0.236805, F1 = 0.924051 (Best Epoch 8)
- **Deficiency**: High false-negative rate on positive reviews due to recency bias.

### B. Round 2 Experiments (T1–T5)
- Explored learning rates (0.001 vs 0.0005), recurrent dropout (0.4 vs 0.5), recurrent units (64 vs 96), and embedding dimensions (64 vs 96).
- **Finding**: Widening recurrent units (96 units, T3/T5) increased validation loss to 0.4185–0.4917 (overfitting). Widening embedding (96-d, T4) added +49% parameters (492,417 weights) without improving validation accuracy (0.9250). The baseline was retained.

### C. Systematic Error Analysis
- Diagnosed two fundamental structural failure modes of the single-state Simple RNN:
  1. **Recency bias**: The final hidden state $h_T$ was overwhelmed by final words; reviews with contrasting concluding sentences were systematically misclassified.
  2. **Information bottleneck**: Compressing 300 words into 64 scalars washed out early sentiment indicators.

### D. Round 3 Experiments (R3-1 to R3-5) & Selection of R3-3
Five targeted architectural candidates were evaluated strictly on the validation set:

| Candidate | Architecture / Change | Val Acc | Val Loss | Val F1 | Selection Decision |
|---|---|:---:|:---:|:---:|---|
| **R3-1** | Baseline + Recurrent $L_2(10^{-4})$ | 0.6062 | 0.6722 | 0.6277 | Underfit; rejected. |
| **R3-2** | Bidirectional SimpleRNN(32) | 0.9125 | 0.2289 | 0.9079 | Lower recall; rejected. |
| **R3-3** | **Sequence-Averaged SimpleRNN(64) + GAP1D** | **0.9438** | **0.137641** | **0.943396** | **Selected Winner (lowest loss, highest F1)** |
| **R3-4** | Baseline with `MAX_LEN=200` | 0.8875 | 0.2887 | 0.8889 | Truncation hurt long reviews; rejected. |
| **R3-5** | Bidirectional SimpleRNN(32) + $L_2$ | 0.9375 | 0.1813 | 0.9367 | Strong, but R3-3 had lower loss and higher F1. |

**Why R3-3 was selected**:
1. **Strongest Generalization**: Validation loss improved from 0.2368 to **0.137641** (**41.9% loss reduction**), and validation accuracy reached **0.9438** (151/160 reviews correct).
2. **Identical Footprint**: Retains the exact 330,369 parameter count as the baseline.
3. **Pure Simple RNN**: Preserves standard un-gated Elman recurrence without adding LSTM/GRU gates.

In [ ]:
history_path = RESULTS / "simple_rnn_history.csv"
if history_path.exists():
    hist_df = pd.read_csv(history_path)
    print("Training History (18 epochs, EarlyStopping restored best epoch 13):")
    print(hist_df[["epoch", "loss", "accuracy", "val_loss", "val_accuracy"]].to_string(index=False))
    
    # Plot learning curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist_df["epoch"], hist_df["loss"], label="Train Loss", color="#1f77b4")
    ax1.plot(hist_df["epoch"], hist_df["val_loss"], label="Val Loss", color="#d62728", linestyle="--")
    ax1.axvline(13, color="gray", linestyle=":", label="Best Epoch (13)")
    ax1.set_title("Simple RNN (R3-3) Loss Curve")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Binary Cross-Entropy")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(hist_df["epoch"], hist_df["accuracy"], label="Train Acc", color="#1f77b4")
    ax2.plot(hist_df["epoch"], hist_df["val_accuracy"], label="Val Acc", color="#2ca02c", linestyle="--")
    ax2.axvline(13, color="gray", linestyle=":", label="Best Epoch (13)")
    ax2.set_title("Simple RNN (R3-3) Accuracy Curve")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print(f"{history_path} not found.")

## 5. Final Test Set Evaluation Results

After completing validation-based model selection, R3-3 was evaluated **once** on the untouched 160-review test set (Seed 42, threshold 0.50).

### Key Test Metrics
| Metric | Baseline Test Result | Final Promoted Model (R3-3) | Delta ($\Delta$) |
|---|:---:|:---:|:---:|
| **Test Accuracy** | 85.00% (136/160) | **95.00%** (152/160) | **+10.00%** (+16 reviews correct) |
| **Test Precision** | 90.00% | **95.00%** | +5.00% |
| **Test Recall** | 78.75% | **95.00%** | **+16.25%** |
| **Test F1 Score** | 0.8400 | **0.9500** | **+0.1100** |
| **Test ROC-AUC** | 0.8961 | **0.979531** | **+0.0834** |
| **Test Loss** | 0.3705 | **0.184034** | **-50.3% loss** |
| **Confusion Matrix** | $\text{TN}=73, \text{FP}=7$<br>$\text{FN}=17, \text{TP}=63$ | **$\text{TN}=76, \text{FP}=4$<br>$\text{FN}=4, \text{TP}=76$** | **Symmetric class performance** |
| **Total Test Errors** | 24 errors | **8 errors** | **66.7% error reduction** |

### 95% Bootstrap Confidence Intervals (1,000 resamples):
- **Accuracy**: $[0.9125, 0.9813]$
- **F1 Score**: $[0.9139, 0.9818]$
- **ROC-AUC**: $[0.9572, 0.9955]$

In [ ]:
metrics_path = RESULTS / "simple_rnn_metrics.json"
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
    print("Official Test Metrics (simple_rnn_metrics.json):")
    print(f"  Model:            {metrics.get('model')}")
    print(f"  Accuracy:         {metrics['accuracy']:.4f} (95% CI: {metrics['bootstrap_95ci']['accuracy'][0]:.4f} - {metrics['bootstrap_95ci']['accuracy'][1]:.4f})")
    print(f"  Precision:        {metrics['precision']:.4f}")
    print(f"  Recall:           {metrics['recall']:.4f}")
    print(f"  F1 Score:         {metrics['f1']:.4f} (95% CI: {metrics['bootstrap_95ci']['f1'][0]:.4f} - {metrics['bootstrap_95ci']['f1'][1]:.4f})")
    print(f"  ROC-AUC:          {metrics['roc_auc']:.6f} (95% CI: {metrics['bootstrap_95ci']['roc_auc'][0]:.4f} - {metrics['bootstrap_95ci']['roc_auc'][1]:.4f})")
    cm = metrics['confusion_matrix']
    print(f"  Confusion Matrix: TN={cm['tn']}, FP={cm['fp']}, FN={cm['fn']}, TP={cm['tp']}")
    print(f"  Total Errors:     {cm['fp'] + cm['fn']} / 160 (5.00% error rate)")
    print(f"  Parameters:       {metrics['trainable_params']:,}")
    print(f"  Training Time:    {metrics['training_time_sec']:.2f} s")
    print(f"  Best Epoch:       {metrics['best_epoch']} / {metrics['epochs_run']}")

## 6. Error Analysis & Generalization Insights

Across the 160 holdout test reviews, the R3-3 model makes only **8 misclassifications** (4 False Positives, 4 False Negatives), down from 24 errors in the baseline.

### Confidence Distribution & Calibration
- **Mean confidence on correct predictions**: **0.9673** (decisive probability separation)
- **Mean confidence on incorrect predictions**: **0.8683** (lower certainty on edge cases)

### Failure Modes Analysis
1. **Rhetorical Complexity & Sarcasm**: Reviews where polite framing disguises negative critique (e.g., subtle irony or backhanded compliments).
2. **Ambiguous / Mixed Feedback**: Reviews detailing both strong positives and severe drawbacks without an overriding sentiment tilt.
3. **Recency Bias Elimination**: By pooling over all token states $(h_1, \dots, h_T)$, R3-3 completely eliminated 16 baseline false negatives caused by positive reviews closing with minor reservations or logistical details.

In [ ]:
preds_path = RESULTS / "simple_rnn_test_predictions.csv"
if preds_path.exists():
    preds_df = pd.read_csv(preds_path)
    errors = preds_df[preds_df["label"] != preds_df["prediction"]].copy()
    print(f"Total misclassifications in test set: {len(errors)} / {len(preds_df)}")
    print(errors[["row_id", "label", "hotel", "deceptive", "prob_positive", "prediction"]].to_string(index=False))
else:
    print(f"{preds_path} not found.")

## 7. Handover Summary for Group Report

The standardized metrics below represent the official Simple RNN entry for the comparative analysis alongside LSTM, GRU, and 1D CNN:

| Model | Test Accuracy | Precision | Recall | F1-Score | ROC-AUC | Parameters | Recurrent Params | Training Time | Best Epoch |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **Simple RNN (R3-3)** | **0.9500** | **0.9500** | **0.9500** | **0.9500** | **0.9795** | **330,369** | **8,256** | **~48 s** | **13** |

### Key Comparative Highlights
- **Recurrent Efficiency**: Simple RNN requires only 8,256 recurrent parameters (compared to 24,960 in GRU and 33,024 in LSTM).
- **Benchmark Competitiveness**: Despite lacking complex gating mechanisms, sequence-averaged Simple RNN achieves 95.00% accuracy and 0.9795 ROC-AUC on this 300-token classification task.

### Deliverables Manifest (`simple_rnn/results/`)
- `simple_rnn_model.keras`: Serialized official trained model checkpoint (330,369 weights)
- `simple_rnn_metrics.json`: Standardized JSON metrics matching group schema
- `simple_rnn_test_predictions.csv`: Per-review predictions on the 160 test reviews
- `simple_rnn_history.csv`: Per-epoch loss and accuracy log across 18 epochs
- `simple_rnn_confusion_matrix.png`: Test set confusion matrix
- `simple_rnn_roc_curve.png`: Test set ROC curve
- `simple_rnn_accuracy_curve.png` & `simple_rnn_loss_curve.png`: Learning curves

## 8. Conclusions & Architectural Insights

1. **Recency Bias was the Bottleneck**: The primary limitation of the traditional Elman RNN on long hotel reviews was not parameter capacity, but temporal recency bias caused by relying exclusively on the final state $h_T$.
2. **Sequence Averaging is Highly Effective**: Incorporating `GlobalAveragePooling1D` allows every token in the 300-step sequence to contribute to document representation, yielding a +10.00% accuracy jump (85.00% → 95.00%) and 66.7% error reduction without increasing parameter count.
3. **Genuine Simple RNN Preserved**: The architecture remains an authentic un-gated recurrent neural network, fulfilling the project requirement while delivering robust generalization.

---
*Notebook 07: Simple RNN Official Model (R3-3)*